# Google Gemini API — Notebook Panduan Cepat (Embeddings API)

Dokumentasi resmi: **https://ai.google.dev/gemini-api/docs/embeddings**

Notebook langsung-pakai untuk memanggil **Google Gemini Embeddings API** dari
Python menggunakan SDK **`google-genai`** modern (`client.models.embed_content(...)`).

## Yang akan kamu lakukan di sini
1. Instal dependensi
2. Menyiapkan API key
3. Membuat klien
4. Embedding dasar (satu teks)
5. Embedding batch (banyak teks sekaligus)
6. Mengatur dimensi output & task type
7. Menghitung kemiripan (cosine similarity)
8. Embedding lokal (offline) dengan sentence-transformers

## Prasyarat
- API key Google AI Studio: https://aistudio.google.com/apikey
- Python 3.9+

> Simpan key kamu di environment variable bernama `GOOGLE_API_KEY` (atau
> `GEMINI_API_KEY`). Lihat sel penyiapan di bawah.

> **Model yang tersedia (April 2026):** `gemini-embedding-2` dan
> `gemini-embedding-001`. Keduanya menghasilkan vektor **3072 dimensi** secara
> default, bisa dipotong ke 768 / 1536 / 3072 lewat `output_dimensionality`.
> `gemini-embedding-001` mendukung `task_type`; `gemini-embedding-2` tidak,
> jadi tekankan lewat prompt. Ruang embedding keduanya tidak kompatibel —
> jangan campur antar model.


## 1. Install dependensi

Jalankan sel berikut satu kali. Sel ini menginstal SDK Google GenAI resmi.

In [ ]:
# Instal SDK Google GenAI resmi
# !pip install -q google-genai

# Verifikasi impor berhasil
from google import genai
print("google-genai berhasil diimpor:", getattr(genai, "__version__", "versi tidak tersedia"))

## 2. Siapkan API key kamu

SDK membaca key dari `GOOGLE_API_KEY` secara default. Set sekali di shell kamu
(bukan di notebook, supaya tidak menyimpan rahasia di file):

```bash
export GOOGLE_API_KEY="isi-api-key-kamu"
```

Atau tambahkan ke file `.env` lalu muat di sini. Sel ini lebih mengutamakan
environment variable dan baru memakai variabel teks polos **hanya sebagai jalan
terakhir** — environment variable adalah jalur yang paling aman.

In [ ]:
import os

# 1) Utamakan environment variable
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# 2) Opsional: muat dari file .env lokal
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
    except ImportError:
        pass

# 3) Jalan terakhir HANYA untuk demo cepat — gunakan env var di produksi
if not api_key:
    api_key = "ISI_API_KEY_KAMU_DI_SINI"

if api_key == "ISI_API_KEY_KAMU_DI_SINI" or not api_key:
    print("PERINGATAN: API key tidak ditemukan. Set GOOGLE_API_KEY di lingkungan kamu.")
else:
    print("API key dimuat ({} karakter)".format(len(api_key)))

## 3. Buat klien

Untuk embeddings kita pakai model `gemini-embedding-001`. Ganti nama model sesuai
kebutuhan kamu (mis. `gemini-embedding-2`).

In [ ]:
from google import genai

# Klien otomatis mengambil GOOGLE_API_KEY dari environment jika ada.
api_key=""
client = genai.Client(api_key=api_key)

MODEL = "gemini-embedding-001"
print("Klien siap:", client)

## 4. Embedding dasar

Panggilan paling sederhana: satu teks masuk, satu vektor keluar. Hasilnya ada di
`result.embeddings[0].values`, berupa daftar `float` sepanjang dimensi model.

In [14]:
result = client.models.embed_content(
    model=MODEL,
    contents="Bagaimana cara kerja AI?",
)

embedding = result.embeddings[0]
print("Dimensi vektor:", len(embedding.values))
print("10 nilai pertama:", embedding.values[:10])
print("Statistik:", embedding.statistics)

Dimensi vektor: 3072
10 nilai pertama: [-0.0035503062, 0.020955442, 0.0056467094, -0.073190905, -0.030032733, 0.007493185, 0.0028464017, 0.0022006258, 0.012508023, -0.0047882656]
Statistik: None


## 5. Embedding batch

Kirim daftar teks sekaligus (banyak `str` di `contents`), dan ambil vektor
per elemen. Gunakan `output_dimensionality` untuk mengecilkan ukuran vektor
karena default-nya 3072.

In [13]:
from google.genai import types

teks = [
    "Apa itu machine learning?",
    "Apa itu deep learning?",
    "Cara membuat kopi yang enak.",
]

result = client.models.embed_content(
    model=MODEL,
    contents=teks,
    config=types.EmbedContentConfig(output_dimensionality=768),
)

for i, emb in enumerate(result.embeddings):
    print("Teks {} -> dimensi {}".format(i, len(emb.values)))
    print("  ", teks[i][:40], "...")
    print("   5 nilai pertama:", emb.values[:5])

Teks 0 -> dimensi 768
   Apa itu machine learning? ...
   5 nilai pertama: [-0.01714742, 0.013511639, -0.002489731, -0.07431768, -0.024492787]
Teks 1 -> dimensi 768
   Apa itu deep learning? ...
   5 nilai pertama: [-0.005718504, -0.0019286453, -0.011752068, -0.08011072, -0.022362204]
Teks 2 -> dimensi 768
   Cara membuat kopi yang enak. ...
   5 nilai pertama: [0.0012100241, 0.030981807, 0.0141703235, -0.060210306, 0.005246011]


## 6. Task type & dimensi output

Gunakan `task_type` untuk menyimpan konteks pemakaian (mis. `SEMANTIC_SIMILARITY`,
`RETRIEVAL_DOCUMENT`, `RETRIEVAL_QUERY`, `CLUSTERING`). Ini membantu model
menghasilkan embedding yang lebih relevan untuk keperluan tersebut. `task_type`
didukung oleh `gemini-embedding-001` (tidak oleh `gemini-embedding-2`).

> Catatan: kalau kamu pakai dimensi output non-3072 pada `gemini-embedding-001`,
> vektor **tidak** otomatis ternormalisasi. Lakukan normalisasi manual bila perlu.

In [12]:
result = client.models.embed_content(
    model=MODEL,
    contents=[
        "Dokumen: kewajiban pelaporan pajak",
        "Dokumen: cara mengajukan permohonan izin",
    ],
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        title="Dokumen legal",
        output_dimensionality=1536,
    ),
)

for emb in result.embeddings:
    print("Dimensi:", len(emb.values), "| nilai pertama:", emb.values[:3])

Dimensi: 1536 | nilai pertama: [-0.020478534, 0.02641484, -0.017729446]
Dimensi: 1536 | nilai pertama: [-0.013471206, 0.013034453, -0.006033688]


## 7. Kemiripan (cosine similarity)

Embedding berguna untuk membandingkan makna antar teks. Contoh di bawah
menghitung kemiripan kosinus antara embedding setiap pasangan teks — nilai 1 =
bermakna mirip, 0 = tidak berhubungan.

```python
import math
def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot / (na * nb)
```

In [11]:
import math

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb)

pasangan = [
    ("Apa itu machine learning?", "Apa itu deep learning?"),
    ("Apa itu machine learning?", "Cara membuat kopi yang enak."),
]

teks = [t for p in pasangan for t in p]
result = client.models.embed_content(model=MODEL, contents=teks)

vecs = [e.values for e in result.embeddings]
for (a, b), idx in zip(pasangan, range(0, len(pasangan) * 2, 2)):
    print("cosine({!r}, {!r}) = {:.3f}".format(a, b, cosine(vecs[idx], vecs[idx + 1])))

cosine('Apa itu machine learning?', 'Apa itu deep learning?') = 0.826
cosine('Apa itu machine learning?', 'Cara membuat kopi yang enak.') = 0.574


## 8. Embedding lokal (offline) dengan sentence-transformers

Alternatif yang sepenuhnya lokal: pakai library **sentence-transformers** dari
Hugging Face. Model `sentence-transformers/all-MiniLM-L6-v2` menghasilkan vektor
**384 dimensi**, ringan, cepat, dan bisa jalan tanpa API key.

> Embedding dari model lokal ini **tidak kompatibel** dengan embedding Gemini —
> ruang vektornya berbeda. Jangan bandingkan antar keduanya.

> Prasyarat: `pip install sentence-transformers` (sekali saja) — sudah tersedia
> di venv project ini (versi 6.0.1 + torch 2.14).

In [8]:
# Instal sentence-transformers (sekali saja)
# !pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
print("sentence-transformers berhasil diimpor")

sentence-transformers berhasil diimpor


In [9]:
from sentence_transformers import SentenceTransformer

# Muat model (diunduh otomatis dari Hugging Face Hub pada pemakaian pertama)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

teks = [
    "Apa itu machine learning?",
    "Apa itu deep learning?",
    "Cara membuat kopi yang enak.",
]

# encode dengan normalisasi L2 otomatis
vecs = model.encode(teks, normalize_embeddings=True)
print("Bentuk tensor:", vecs.shape)   # (3, 384)
print("Dimensi per teks:", vecs.shape[1])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2549.38it/s]


Bentuk tensor: (3, 384)
Dimensi per teks: 384


In [10]:
import numpy as np

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pasangan = [
    ("Apa itu machine learning?", "Apa itu deep learning?"),
    ("Apa itu machine learning?", "Cara membuat kopi yang enak."),
]

teks = [t for p in pasangan for t in p]
vecs = model.encode(teks, normalize_embeddings=True)
for (a, b), idx in zip(pasangan, range(0, len(pasangan) * 2, 2)):
    print("cosine({!r}, {!r}) = {:.3f}".format(a, b, cos(vecs[idx], vecs[idx + 1])))

cosine('Apa itu machine learning?', 'Apa itu deep learning?') = 0.859
cosine('Apa itu machine learning?', 'Cara membuat kopi yang enak.') = 0.146


## Selanjutnya

- Eksplorasi model `gemini-embedding-2` untuk input gambar/audio/video/PDF.
- Simpan embedding ke database vektor (mis. FAISS, Chroma, Vertex AI Vector Search).
- Untuk batch besar, gunakan Batch API agar lebih hemat.

> Dokumentasi resmi: [Embeddings | Gemini API](https://ai.google.dev/gemini-api/docs/embeddings)
> dan model [text-embedding](https://ai.google.dev/api/embeddings).